# Analytics Layer - DuckDB SQL Query Execution

This notebook loads the Gold layer star schema tables into DuckDB and executes analytical SQL queries to generate business insights.

## 1. Import Required Libraries

In [29]:
import duckdb
import pandas as pd
from pathlib import Path
import json
from datetime import datetime

# Setup paths
work_dir = Path('/home/jovyan/work')
gold_dir = work_dir / 'output' / 'gold'
submission_dir = Path('/home/jovyan/work')

print(f"Gold layer directory: {gold_dir}")
print(f"Files available: {list(gold_dir.glob('*.csv'))}")

Gold layer directory: /home/jovyan/work/output/gold
Files available: [PosixPath('/home/jovyan/work/output/gold/dim_customers.csv'), PosixPath('/home/jovyan/work/output/gold/fact_order_items.csv'), PosixPath('/home/jovyan/work/output/gold/dim_sellers.csv'), PosixPath('/home/jovyan/work/output/gold/dim_products.csv')]


## 2. Connect to DuckDB and Load Gold Layer Tables

In [30]:
# Connect to DuckDB and load tables
con = duckdb.connect(':memory:')
print("Connected to DuckDB")

tables = ['fact_order_items', 'dim_customers', 'dim_products', 'dim_sellers']
table_stats = {}

for table_name in tables:
    csv_path = gold_dir / f'{table_name}.csv'
    if csv_path.exists():
        con.execute(f"CREATE TABLE {table_name} AS SELECT * FROM read_csv_auto('{csv_path}')")
        result = con.execute(f"SELECT COUNT(*) as row_count FROM {table_name}").fetchall()
        row_count = result[0][0]
        table_stats[table_name] = row_count
        print(f"Loaded {table_name}: {row_count} rows")
    else:
        print(f"Warning: {csv_path} not found")

Connected to DuckDB
Loaded fact_order_items: 1000 rows
Loaded dim_customers: 1000 rows
Loaded dim_products: 1000 rows
Loaded dim_sellers: 1000 rows


## 3. Query 1: Revenue Trend Analysis

In [31]:
# Load SQL query
query_file = submission_dir / 'sql' / 'query_1_revenue_trend_analysis.sql'

if query_file.exists():
    with open(query_file, 'r') as f:
        query = f.read()
    print(f"Loaded query from {query_file}")
else:
    print(f"Error: Query file not found: {query_file}")

Loaded query from /home/jovyan/work/sql/query_1_revenue_trend_analysis.sql


In [32]:
# Execute query
try:
    result = con.execute(query).fetchall()
    columns = [desc[0] for desc in con.description]
    df_results = pd.DataFrame(result, columns=columns)
    
    print(f"Query executed: {len(df_results)} rows")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', None)
    print(df_results)
    
except Exception as e:
    print(f"Error: {str(e)}")

Query executed: 2 rows
  order_month  total_orders  total_items_sold  gross_revenue  \
0  2018-07-01            27                28        3516.33   
1  2018-08-01            29                31        4331.24   

   total_payment_received  avg_item_value  avg_delivery_days  \
0                     NaN      125.583214           7.142857   
1                  138.84      139.717419           8.366667   

   late_delivery_rate_pct  unique_customers  unique_products_sold  \
0                    0.00                 0                     2   
1                    3.23                 0                     2   

   unique_sellers  
0              12  
1              12  


## 4. Query 2: Seller Performance Scorecard

In [33]:
# Load query 2
query_2_file = submission_dir / 'sql' / 'query_2_seller_performance_scorecard.sql'

if query_2_file.exists():
    with open(query_2_file, 'r') as f:
        query_2 = f.read()
    print(f"Loaded query from {query_2_file}")
else:
    print(f"Error: Query file not found: {query_2_file}")

Loaded query from /home/jovyan/work/sql/query_2_seller_performance_scorecard.sql


In [34]:
# Execute query 2
try:
    result_2 = con.execute(query_2).fetchall()
    columns_2 = [desc[0] for desc in con.description]
    df_results_2 = pd.DataFrame(result_2, columns=columns_2)
    
    print(f"Query executed: {len(df_results_2)} rows")
    pd.set_option('display.max_columns', None)
    print(df_results_2)
    
except Exception as e:
    print(f"Error: {str(e)}")

Query executed: 22 rows
    seller_key                         seller_id            seller_city  \
0          594  9c4d31c7e46ab03a43fc06e3142afd4e         rio de janeiro   
1          311  53243585a1d6dc2643021fd1853d8905       lauro de freitas   
2          334  59b22a78efb79a4797979612b885db36             uberlandia   
3          784  ceaec5548eefc6e23e6607c5435102e7              sao paulo   
4          378  6560211a19b47992c3666cc44a7e94c0              sao paulo   
5          476  7b07b3c7487f0ea825fc6df75abd658b              sao paulo   
6            7  0176f73cc1195f367f7b32db1e5b3aa8               ibitinga   
7          453  751bdc4d83a466c7206cd42e8f426b03         ribeirao pires   
8          988  fde0cc9ea29c8ccfc0a2c22256a58c71               curitiba   
9          892  e9779976487b77c6d4ac45f75ec7afe9           praia grande   
10         717  c013e57c075a06e5b5c48ee03c525719             sao carlos   
11         431  7142540dd4c91e2237acb7e911c4eba2              penapolis   
1

## 5. Schema Verification - Gold Layer Tables

In [35]:
# Display table schemas
for table_name in tables:
    try:
        schema_result = con.execute(f"DESCRIBE {table_name}").fetchall()
        print(f"\n{table_name}:")
        for col_info in schema_result:
            col_name, col_type = col_info[0], col_info[1]
            print(f"  {col_name}: {col_type}")
    except Exception as e:
        print(f"Error describing {table_name}: {str(e)}")


fact_order_items:
  order_item_sk: BIGINT
  order_id: VARCHAR
  order_item_id: BIGINT
  customer_key: VARCHAR
  product_key: BIGINT
  seller_key: BIGINT
  order_date: DATE
  order_status: VARCHAR
  price: DOUBLE
  freight_value: DOUBLE
  payment_value: DOUBLE
  payment_type: VARCHAR
  payment_installments_max: BIGINT
  days_to_deliver: BIGINT
  days_delivery_vs_estimate: BIGINT
  is_late_delivery: BOOLEAN

dim_customers:
  customer_key: BIGINT
  customer_unique_id: VARCHAR
  customer_city: VARCHAR
  customer_state: VARCHAR
  customer_zip_code_prefix: BIGINT
  first_order_date: TIMESTAMP WITH TIME ZONE
  total_orders: BIGINT
  total_spend: DOUBLE
  is_repeat_customer: BOOLEAN

dim_products:
  product_key: BIGINT
  product_id: VARCHAR
  product_category_name: VARCHAR
  product_weight_g: DOUBLE
  product_volume_cm3: DOUBLE
  product_photos_qty: BIGINT
  product_description_lenght: BIGINT

dim_sellers:
  seller_key: BIGINT
  seller_id: VARCHAR
  seller_city: VARCHAR
  seller_state: VARCHA

## 6. Data Quality Summary

In [36]:
print("Data Quality Summary:\n")

for table_name in tables:
    try:
        count_result = con.execute(f"SELECT COUNT(*) as total FROM {table_name}").fetchall()
        total_rows = count_result[0][0]
        
        schema_result = con.execute(f"DESCRIBE {table_name}").fetchall()
        columns = [col[0] for col in schema_result]
        
        print(f"{table_name}: {total_rows} rows, {len(columns)} columns")
        
    except Exception as e:
        print(f"Error: {str(e)}")

print("\nDone.")

Data Quality Summary:

fact_order_items: 1000 rows, 16 columns
dim_customers: 1000 rows, 9 columns
dim_products: 1000 rows, 7 columns
dim_sellers: 1000 rows, 5 columns

Done.
